# Minecraft RL Agent - Colab Training

One-click training notebook for Google Colab (free T4 GPU).

**What this does:**
1. Installs MineRL and dependencies
2. Mounts Google Drive for checkpoint persistence
3. Trains PPO on MineRLTreechop-v0
4. Auto-checkpoints every 50K steps (survives Colab disconnects)
5. Logs to TensorBoard

**Expected runtime:** ~5M steps per 12-hour Colab session

In [ ]:
# Step 1: Clone repo and install dependencies
import os

if not os.path.exists('minecraft-ai'):
    !git clone https://github.com/Bobminion21/minecraft-ai.git

os.chdir('minecraft-ai')
!pip install -e .
!pip install minerl==0.4.4

In [ ]:
# Step 2: Mount Google Drive for persistent checkpoints
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/minecraft-ai'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/videos', exist_ok=True)
print(f'Drive mounted. Checkpoints: {DRIVE_DIR}/checkpoints')

In [ ]:
# Step 3: Verify GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Step 4: Configure training
from minecraft_ai.utils.config import Config

config = Config(
    # Environment
    env_id='MineRLTreechop-v0',
    frame_size=64,
    frame_stack=4,
    max_episode_steps=8000,
    
    # Training
    total_timesteps=5_000_000,  # ~12 hours on T4
    rollout_length=128,
    ppo_epochs=4,
    num_minibatches=4,
    
    # Checkpointing (to Google Drive!)
    checkpoint_interval=50_000,
    checkpoint_dir=f'{DRIVE_DIR}/checkpoints',
    log_dir=f'{DRIVE_DIR}/logs',
    video_dir=f'{DRIVE_DIR}/videos',
    
    # Standard PPO hyperparams
    learning_rate=2.5e-4,
    gamma=0.99,
    gae_lambda=0.95,
    clip_epsilon=0.1,
    entropy_coeff=0.01,
    
    seed=42,
)

print(f'Config ready. Training for {config.total_timesteps:,} steps.')
print(f'Checkpoints every {config.checkpoint_interval:,} steps to Google Drive.')

In [ ]:
# Step 5: Create MineRL environment
# NOTE: First run downloads Minecraft assets (~1GB), takes a few minutes
import minerl
import gymnasium as gym
from minecraft_ai.envs.wrappers import wrap_env

base_env = gym.make(config.env_id)
env = wrap_env(config, env=base_env)
print(f'Environment ready: {config.env_id}')
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space}')

In [ ]:
# Step 6: Start training!
from minecraft_ai.training.trainer import Trainer

trainer = Trainer(config, env=env)
trainer.train()

In [ ]:
# Step 7: Launch TensorBoard (optional)
%load_ext tensorboard
%tensorboard --logdir {DRIVE_DIR}/logs

In [ ]:
# Step 8: Record gameplay video (after training)
from minecraft_ai.evaluation.video_recorder import VideoRecorder
from minecraft_ai.utils.torch_utils import get_device

recorder = VideoRecorder(env, trainer.model, get_device(), config.video_dir)
recorder.record('trained_agent.mp4', max_steps=2000)

In [ ]:
# Step 9: Plot training curves
from minecraft_ai.evaluation.plotting import plot_training_curves

csv_path = f'{DRIVE_DIR}/logs/run/metrics.csv'
plot_training_curves(csv_path, output_path=f'{DRIVE_DIR}/training_curves.png')